# 0 — Install dependencies & authenticate

In [ ]:
# Install earthengine API (pre-installed on Colab but pin version for safety)
!pip install -q earthengine-api==0.1.390 geemap

import ee
import geemap
import os, json, shutil
import numpy as np

# ── Authenticate GEE ──────────────────────────────────────────────────────
# This opens a browser link — sign in with your Google account and paste
# the authorization code back here. Only needed once per session.
ee.Authenticate()
ee.Initialize(project='replace with your GEE project ID') 
print('GEE authenticated ✅')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.6/314.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.4 MB/s eta 0:00:00


GEE authenticated ✅


In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Verify the FloodProject_Assam2023 folder exists
DRIVE_BASE = '/content/drive/MyDrive/FloodProject_Assam2023'
os.makedirs(DRIVE_BASE, exist_ok=True)

# Local scratch space (ephemeral, fast)
LOCAL_BASE = '/content/assam2023'
os.makedirs(LOCAL_BASE, exist_ok=True)

print(f'Drive mounted ✅')
print(f'Drive base : {DRIVE_BASE}')
print(f'Local base : {LOCAL_BASE}')

Mounted at /content/drive
Drive mounted ✅
Drive base : /content/drive/MyDrive/FloodProject_Assam2023
Local base : /content/assam2023


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
DRIVE_BASE = '/content/drive/MyDrive/FloodProject_Assam2023'
files = sorted(os.listdir(DRIVE_BASE))
total_gb = sum(os.path.getsize(f'{DRIVE_BASE}/{f}') for f in files) / 1e9
print(f'Total files: {len(files)}  Total size: {total_gb:.1f} GB')
for f in files:
    size = os.path.getsize(f'{DRIVE_BASE}/{f}') / 1e6
    print(f'  {f:<60} {size:.1f} MB')

Total files: 49  Total size: 31.9 GB
  Assam2023_TEMPORAL.tif                                       0.1 MB
  assam2023_DEM-0000000000-0000000000.tif                      456.5 MB
  assam2023_DEM-0000000000-0000016384.tif                      587.4 MB
  assam2023_DEM-0000000000-0000032768.tif                      757.8 MB
  assam2023_DEM-0000000000-0000049152.tif                      1304.7 MB
  assam2023_DEM-0000000000-0000065536.tif                      229.9 MB
  assam2023_DEM-0000016384-0000000000.tif                      910.9 MB
  assam2023_DEM-0000016384-0000016384.tif                      1064.5 MB
  assam2023_DEM-0000016384-0000032768.tif                      1848.0 MB
  assam2023_DEM-0000016384-0000049152.tif                      33.2 MB
  assam2023_DEM-0000016384-0000065536.tif                      5.1 MB
  assam2023_DEM-0000032768-0000000000.tif                      10.2 MB
  assam2023_DEM-0000032768-0000016384.tif                      364.8 MB
  assam2023_DEM-0000032768-000

# PART A — GEE Reference Mask Generation

**Method:** S1 VH threshold + S2 MNDWI fusion → identical to Sen1Floods11 weak label approach

**Target date:** ~29 August 2023 (Wave 3 peak — flooding worsened from Aug 27)

**Label convention (Sen1Floods11):**
- `1` = flood water
- `0` = no water / land
- `-1` = invalid (cloud gap AND no S1 coverage)

**Where S2 is cloudy, S1 fills the gap → very few -1 pixels expected.**

In [ ]:
# ── Define Assam AOI ──────────────────────────────────────────────────────
states = ee.FeatureCollection('FAO/GAUL/2015/level1')
assam  = states.filter(ee.Filter.eq('ADM1_NAME', 'Assam'))
aoi    = assam.geometry()

print('Assam AOI loaded ✅')
print(f'Bounds: {aoi.bounds().getInfo()["coordinates"]}')

Assam AOI loaded ✅
Bounds: [[[89.6948399758636, 24.13475067990437], [96.02091316406238, 24.13475067990437], [96.02091316406238, 27.977372102348333], [89.6948399758636, 27.977372102348333], [89.6948399758636, 24.13475067990437]]]


In [ ]:
# ── SENTINEL-1 branch ─────────────────────────────────────────────────────
# S1 is cloud-immune — use it to fill gaps where S2 is cloudy
# Date window: ±6 days around Aug 29 (S1 revisit ~12 days over Assam)

s1_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
  .filterBounds(aoi)
  .filterDate('2023-08-23', '2023-09-05')
  .filter(ee.Filter.eq('instrumentMode', 'IW'))
  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
  .select(['VH', 'VV']))

print(f'S1 scenes found: {s1_col.size().getInfo()}')

# Speckle reduction: focal mean 50m radius (same as Sen1Floods11)
def speckle_filter(img):
    vh = img.select('VH').focal_mean(radius=50, kernelType='circle', units='meters')
    vv = img.select('VV').focal_mean(radius=50, kernelType='circle', units='meters')
    return (img
        .addBands(vh.rename('VH_sm'))
        .addBands(vv.rename('VV_sm')))

s1_filtered = s1_col.map(speckle_filter)
s1_img      = s1_filtered.median().clip(aoi)

# S1 water mask: VH < -16 dB (standard open water threshold for S1 IW)
# Tune if needed: -15 dB = more permissive, -18 dB = stricter
S1_VH_THRESHOLD = -16
s1_water = s1_img.select('VH_sm').lt(S1_VH_THRESHOLD).rename('s1_water')

# Track where S1 has valid data (should be everywhere)
s1_valid = s1_col.select('VH').count().gt(0).rename('s1_valid').clip(aoi)

print(f'S1 water mask created (VH < {S1_VH_THRESHOLD} dB) ✅')

S1 scenes found: 15
S1 water mask created (VH < -16 dB) ✅


In [ ]:
# ── SENTINEL-2 branch ─────────────────────────────────────────────────────
# S2 gives cleaner water delineation where cloud-free
# ±10 day window — wide enough to find some clear pixels over each area

def mask_s2_clouds(img):
    """SCL-based cloud mask for S2 SR. Keeps: vegetation, bare soil, water, snow."""
    scl   = img.select('SCL')
    clear = (scl.eq(4)   # vegetation
              .Or(scl.eq(5))   # bare soil
              .Or(scl.eq(6))   # water
              .Or(scl.eq(11))) # snow
    return (img
        .updateMask(clear)
        .divide(10000)  # DN → reflectance [0,1]
        .copyProperties(img, ['system:time_start']))

s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(aoi)
  .filterDate('2023-08-19', '2023-09-08')
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))  # keep partially cloudy
  .map(mask_s2_clouds))

print(f'S2 scenes found: {s2_col.size().getInfo()}')

s2_median = s2_col.median().clip(aoi)

# MNDWI = (Green - SWIR1) / (Green + SWIR1) = (B3 - B11) / (B3 + B11)
# Water pixels: MNDWI > 0.0 (tune: 0.0 is standard, lower = more water detected)
mndwi   = s2_median.normalizedDifference(['B3', 'B11']).rename('MNDWI')
s2_water = mndwi.gt(0.0).rename('s2_water')

# Track where S2 has valid (cloud-free) observations
s2_valid = s2_col.select('B3').count().gt(0).rename('s2_valid').clip(aoi)

# Quick coverage check
cloud_gap = (s2_valid.eq(0)
  .reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e10)
  .getInfo())
print(f'S2 cloud gap fraction: {cloud_gap["s2_valid"]:.3f}')
print('(0 = all clear, 1 = completely cloudy — S1 fills the gap either way)')

S2 scenes found: 68
S2 cloud gap fraction: 0.000
(0 = all clear, 1 = completely cloudy — S1 fills the gap either way)


In [ ]:
# ── FUSION: S1 + S2 ───────────────────────────────────────────────────────
#
# Priority rule (matches Sen1Floods11 methodology):
#   1. Where S2 is cloud-free → use S2 MNDWI water (more accurate)
#   2. Where S2 is cloudy     → fall back to S1 VH water (cloud-immune)
#   3. Where NEITHER has data → mark as -1 (invalid)

# S2 where valid, S1 where S2 is cloudy
fused_water = s2_water.where(s2_valid.eq(0), s1_water).rename('fused_water')

# ── Remove permanent water (JRC Global Surface Water) ─────────────────────
# We want FLOOD ONLY — not rivers/lakes that are permanently water
jrc       = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
perm_water = jrc.select('seasonality').gte(10)  # water ≥10 months/yr = permanent
flood_only = fused_water.where(perm_water, ee.Image(0)).rename('flood_only')

# ── Build final label mask (Sen1Floods11 convention: 1 / 0 / -1) ──────────
any_valid = s2_valid.Or(s1_valid)  # at least one sensor has data

label = (ee.Image(-1)               # default: invalid
  .where(any_valid, flood_only)     # fill with flood mask where any data exists
  .rename('label')
  .clip(aoi)
  .toInt8())

# ── Stats ─────────────────────────────────────────────────────────────────
flood_frac = (flood_only.eq(1)
  .reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e10)
  .getInfo())
invalid_frac = (label.eq(-1)
  .reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e10)
  .getInfo())

print(f'Flood fraction  : {flood_frac["flood_only"]:.4f}')
print(f'Invalid fraction: {invalid_frac["label"]:.4f}  (should be near 0 due to S1 fill)')
print('Label mask built ✅')

Flood fraction  : 0.0915
Invalid fraction: 0.0701  (should be near 0 due to S1 fill)
Label mask built ✅


In [ ]:
# Check where -1 pixels are spatially
invalid_count = label.eq(-1).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'Total invalid pixels (at 1km scale): {invalid_count}')

# Check if they're clustered at boundary by eroding the AOI slightly
aoi_inner = aoi.buffer(-5000)  # shrink by 5km inward
invalid_interior = label.eq(-1).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'Invalid fraction in interior (5km from border removed): {invalid_interior}')

Total invalid pixels (at 1km scale): {'label': 6116.854901960781}
Invalid fraction in interior (5km from border removed): {'label': 0.07626909290414627}


In [ ]:
# Check S1 valid coverage separately
s1_invalid = s1_valid.eq(0).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'S1 invalid fraction (interior): {s1_invalid}')

# Check S2 valid coverage separately
s2_invalid = s2_valid.eq(0).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'S2 invalid fraction (interior): {s2_invalid}')

# Check if the fusion logic is working — any_valid should cover everything S1 covers
any_valid_check = any_valid.eq(0).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'any_valid=0 fraction (interior): {any_valid_check}')

S1 invalid fraction (interior): {'s1_valid': 0}
S2 invalid fraction (interior): {'s2_valid': 0}
any_valid=0 fraction (interior): {'s2_valid': 0}


In [ ]:
# Fix: rename both to same band name before OR
any_valid = (s2_valid.rename('valid')
             .Or(s1_valid.rename('valid')))

# Rebuild label with fixed any_valid
label = (ee.Image(-1)
  .where(any_valid, flood_only)
  .rename('label')
  .clip(aoi)
  .toInt8())

# Verify
invalid_fixed = label.eq(-1).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'Invalid fraction after fix: {invalid_fixed}')

Invalid fraction after fix: {'label': 0.07626909290414627}


In [ ]:
# Step by step debug — print each intermediate
# Check 1: does ee.Image(-1) itself have full coverage?
check1 = ee.Image(-1).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'1. ee.Image(-1) mean (should be -1.0): {check1}')

# Check 2: what does any_valid actually look like?
check2 = any_valid.rename('valid').reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'2. any_valid mean (should be ~1.0): {check2}')

# Check 3: what does flood_only_filled look like?
check3 = flood_only_filled.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'3. flood_only_filled mean (should be ~0.09): {check3}')

# Check 4: what does label look like BEFORE .clip(aoi)?
label_unclipped = (ee.Image(-1)
  .where(any_valid, flood_only_filled)
  .rename('label')
  .toInt8())
check4 = label_unclipped.eq(-1).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'4. Invalid fraction before clip (should be ~0): {check4}')

# Check 5: is .clip(aoi) itself introducing the gaps?
check5 = label.eq(-1).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'5. Invalid fraction after clip (current): {check5}')

1. ee.Image(-1) mean (should be -1.0): {'constant': -1}
2. any_valid mean (should be ~1.0): {'valid': 1}
3. flood_only_filled mean (should be ~0.09): {'flood_only': 0.09402535255037776}
4. Invalid fraction before clip (should be ~0): {'label': 0.07626909290414627}
5. Invalid fraction after clip (current): {'label': 0.07626909290414627}


In [ ]:
# Bypass .where() completely — use arithmetic instead
# Logic: label = -1 * (1 - any_valid_int) + flood_only_filled * any_valid_int
# Since any_valid_int = 1 everywhere, this should just = flood_only_filled

label_arith = (flood_only_filled.multiply(any_valid_int)
    .add(ee.Image(-1).multiply(any_valid_int.eq(0)))
    .rename('label')
    .toInt8()
    .clip(aoi))

check_arith = label_arith.eq(-1).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'Invalid fraction (arithmetic): {check_arith}')

# Also try the most direct possible thing
label_direct = flood_only_filled.toInt8().clip(aoi).rename('label')
check_direct = label_direct.eq(-1).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi_inner,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'Invalid fraction (direct flood_only cast): {check_direct}')

Invalid fraction (arithmetic): {'label': 0}
Invalid fraction (direct flood_only cast): {'label': 0}


In [ ]:
# Use arithmetic label (bypasses the .where() bug)
label = label_arith  # already defined above — {0, 1} only, no -1s

# Final verification
stats = label.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=1000,
    maxPixels=1e10
).getInfo()
print(f'Label value counts: {stats}')

Label value counts: {'label': {'0': 73665.77647058845, '1': 7418.29019607843}}


In [ ]:
# ── Visualize (optional sanity check) ─────────────────────────────────────
Map = geemap.Map()
Map.centerObject(aoi, 7)
Map.addLayer(aoi,        {'color': '888888'}, 'Assam boundary')
Map.addLayer(s2_valid,   {'min': 0, 'max': 1, 'palette': ['red', 'green']}, 'S2 valid coverage')
Map.addLayer(mndwi,      {'min': -0.5, 'max': 0.5, 'palette': ['brown', 'white', 'blue']}, 'S2 MNDWI')
Map.addLayer(s1_img.select('VH_sm'), {'min': -25, 'max': 0}, 'S1 VH smoothed')
Map.addLayer(label,      {'min': -1, 'max': 1, 'palette': ['black', 'green', 'blue']}, 'Label (-1=invalid, 0=land, 1=flood)')
Map

Map(center=[26.352060685710285, 92.81483911156315], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# ── Export reference mask to Google Drive ─────────────────────────────────
# This is a large export (10m over Assam ~78,000 km²) — will take 30-60 min
# Check Tasks tab in GEE code editor or use ee.batch.Task.status()

task_label = ee.batch.Export.image.toDrive(
    image          = label_arith,   # ← swap this
    description    = 'Assam_flood_reference_mask_Aug2023',
    folder         = 'FloodProject_Assam2023',
    fileNamePrefix = 'assam_flood_label_s1s2_aug2023',
    region         = aoi,
    scale          = 10,
    crs            = 'EPSG:4326',
    maxPixels      = 1e11,
    fileFormat     = 'GeoTIFF'
)
task_label.start()
print(f'Export started ✅  Task ID: {task_label.id}')

# Also export S2 valid coverage map (useful for reporting cloud gap % in paper)
task_valid = ee.batch.Export.image.toDrive(
    image       = s2_valid.toInt8(),
    description = 'Assam_S2_valid_coverage_Aug2023',
    folder      = 'FloodProject_Assam2023',
    fileNamePrefix = 'assam_s2_valid_coverage_aug2023',
    region      = aoi,
    scale       = 100,   # coarser res fine for coverage diagnostic
    crs         = 'EPSG:4326',
    maxPixels   = 1e11,
    fileFormat  = 'GeoTIFF'
)
task_valid.start()
print(f'Coverage export started ✅  Task ID: {task_valid.id}')

print()
print('Monitor export progress:')
print('  https://code.earthengine.google.com/tasks')
print('Once COMPLETED, continue to Part B below.')

Export started ✅  Task ID: SA4ORULVX4BJIEIV757GAOHX
Coverage export started ✅  Task ID: EIR6VQCOHLY2XA6YAM3PVODN

Monitor export progress:
  https://code.earthengine.google.com/tasks
Once COMPLETED, continue to Part B below.


In [ ]:
import os
drive_base = '/content/drive/MyDrive/FloodProject_Assam2023'
label_tiles = [f for f in os.listdir(drive_base)
               if 'flood_label' in f and f.endswith('.tif')]
print(f'Label tiles found: {len(label_tiles)}')
for f in sorted(label_tiles):
    print(f'  {f}')

Label tiles found: 2
  assam_flood_label_s1s2_aug2023-0000000000-0000000000.tif
  assam_flood_label_s1s2_aug2023-0000000000-0000065536.tif


In [ ]:
import rasterio
import numpy as np

# Read first tile
tile_path = f'{drive_base}/{sorted(label_tiles)[0]}'
with rasterio.open(tile_path) as src:
    print(f'CRS       : {src.crs}')
    print(f'Resolution: {src.res}')
    print(f'Shape     : {src.height}×{src.width}')
    print(f'Dtype     : {src.dtypes}')
    chip = src.read(1)
    vals, counts = np.unique(chip, return_counts=True)
    for v, c in zip(vals, counts):
        print(f'  value {v:>3}: {c:>10,} pixels')

CRS       : EPSG:4326
Resolution: (8.983152841195215e-05, 8.983152841195215e-05)
Shape     : 42777×65536
Dtype     : ('int8',)
  value   0: 2,727,446,691 pixels
  value   1: 75,986,781 pixels


In [ ]:
from osgeo import gdal

vrt_label = f'/content/assam2023/label.vrt'
gdal.BuildVRT(vrt_label,
    [f'{drive_base}/{f}' for f in sorted(label_tiles)])

with rasterio.open(vrt_label) as src:
    print(f'Full VRT shape : {src.height}×{src.width}')
    print(f'Full VRT bounds: {src.bounds}')

/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Full VRT shape : 42777×70422
Full VRT bounds: BoundingBox(left=89.69480482570916, bottom=24.134677412325537, right=96.02092071953565, top=27.977400703203614)


In [ ]:
# ── Assam 2023 Data Re-export ─────────────────────────────────
# Matches original pipeline exactly:
#   DEM: Copernicus GLO-30, bands = Elevation/Slope/TWI/HAND, scale=30m→10m
#   Temporal: CHIRPS + ERA5, interleaved 30 bands, scale=10000m
#   S1: VV+VH median composite
#   S2: B2,B3,B4,B8,B11,B12 cloud-masked median

import ee, datetime

# GEE already authenticated from Part A
# Assam AOI already defined as `aoi`

DRIVE_FOLDER = 'FloodProject_Assam2023'
FLOOD_DATE   = '2023-08-29'
WINDOW_DAYS  = 15

# ── DEM (Elevation, Slope, TWI, HAND) ─────────────────────────
# Exactly as in 01_gee_export_dem_derivatives.ipynb
dem   = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic()
slope = ee.Terrain.slope(dem).rename('Slope')

slope_rad      = slope.multiply(3.14159265 / 180.0)
tan_slope      = slope_rad.tan()
tan_slope_safe = tan_slope.where(tan_slope.lt(0.001), 0.001)
flow_acc       = dem.reduceNeighborhood(
    reducer = ee.Reducer.sum(),
    kernel  = ee.Kernel.circle(radius=5, units='pixels')
)
twi       = flow_acc.divide(tan_slope_safe).log().rename('TWI')
focal_min = dem.focalMin(radius=50, units='pixels')
hand      = dem.subtract(focal_min).rename('HAND')

dem_stack = (dem.rename('Elevation')
               .addBands(slope)
               .addBands(twi)
               .addBands(hand)
               .toFloat()
               .clip(aoi))

task_dem = ee.batch.Export.image.toDrive(
    image          = dem_stack,
    description    = 'Assam2023_DEM',
    folder         = DRIVE_FOLDER,
    fileNamePrefix = 'assam2023_DEM',
    region         = aoi,
    scale          = 10,
    crs            = 'EPSG:4326',
    maxPixels      = int(1e13),
    fileFormat     = 'GeoTIFF'
)
task_dem.start()
print(f'DEM export started ✅  ID: {task_dem.id}')

# ── S1 FLOOD ──────────────────────────────────────────────────
s1_flood = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2023-08-23', '2023-09-05')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .select(['VV', 'VH'])
    .median()
    .toFloat()
    .clip(aoi))

task_s1f = ee.batch.Export.image.toDrive(
    image          = s1_flood,
    description    = 'Assam2023_S1_flood',
    folder         = DRIVE_FOLDER,
    fileNamePrefix = 'assam2023_S1_flood',
    region         = aoi,
    scale          = 10,
    crs            = 'EPSG:4326',
    maxPixels      = int(1e13),
    fileFormat     = 'GeoTIFF'
)
task_s1f.start()
print(f'S1 flood export started ✅  ID: {task_s1f.id}')

# ── S1 PRE-FLOOD ──────────────────────────────────────────────
s1_pre = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2023-01-01', '2023-05-31')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .select(['VV', 'VH'])
    .median()
    .toFloat()
    .clip(aoi))

task_s1p = ee.batch.Export.image.toDrive(
    image          = s1_pre,
    description    = 'Assam2023_S1_pre',
    folder         = DRIVE_FOLDER,
    fileNamePrefix = 'assam2023_S1_pre',
    region         = aoi,
    scale          = 10,
    crs            = 'EPSG:4326',
    maxPixels      = int(1e13),
    fileFormat     = 'GeoTIFF'
)
task_s1p.start()
print(f'S1 pre-flood export started ✅  ID: {task_s1p.id}')

# ── S2 (B2,B3,B4,B8,B11,B12) ─────────────────────────────────
# mask_s2_clouds already defined from Part A
s2_export = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate('2023-08-19', '2023-09-08')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
    .map(mask_s2_clouds)
    .select(['B2','B3','B4','B8','B11','B12'])
    .median()
    .multiply(10000).toInt16()
    .clip(aoi))

task_s2 = ee.batch.Export.image.toDrive(
    image          = s2_export,
    description    = 'Assam2023_S2',
    folder         = DRIVE_FOLDER,
    fileNamePrefix = 'assam2023_S2',
    region         = aoi,
    scale          = 10,
    crs            = 'EPSG:4326',
    maxPixels      = int(1e13),
    fileFormat     = 'GeoTIFF'
)
task_s2.start()
print(f'S2 export started ✅  ID: {task_s2.id}')

# ── TEMPORAL (CHIRPS + ERA5, 30 bands interleaved) ────────────
# Exactly as in 02_gee_export_temporal.ipynb
def date_minus(date_str, days):
    d = datetime.date.fromisoformat(date_str)
    return str(d - datetime.timedelta(days=days))

chirps_col = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
era5_col   = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')

bands, bnames = [], []
for i in range(WINDOW_DAYS):
    day_offset = WINDOW_DAYS - 1 - i
    day_date   = date_minus(FLOOD_DATE, day_offset)
    next_date  = date_minus(FLOOD_DATE, day_offset - 1)

    chirps_day = (chirps_col
        .filterDate(day_date, next_date)
        .select('precipitation')
        .mosaic()
        .rename(f'chirps_day{i:02d}')
        .toFloat())

    era5_day = (era5_col
        .filterDate(day_date, next_date)
        .select('volumetric_soil_water_layer_1')
        .mosaic()
        .rename(f'era5_day{i:02d}')
        .toFloat())

    bands.extend([chirps_day, era5_day])
    bnames.extend([f'chirps_day{i:02d}', f'era5_day{i:02d}'])

temporal_stack = ee.Image.cat(*bands).rename(bnames).clip(aoi)

task_temp = ee.batch.Export.image.toDrive(
    image          = temporal_stack,
    description    = 'Assam2023_TEMPORAL',
    folder         = DRIVE_FOLDER,
    fileNamePrefix = 'Assam2023_TEMPORAL',
    region         = aoi,
    scale          = 10000,   # 10km — exactly as original
    crs            = 'EPSG:4326',
    maxPixels      = int(1e9),
    fileFormat     = 'GeoTIFF',
    formatOptions  = {'cloudOptimized': True}
)
task_temp.start()
print(f'Temporal export started ✅  ID: {task_temp.id}')

print()
print('All 5 exports running in parallel ✅')
print('Monitor: https://code.earthengine.google.com/tasks')
print()
print('Expected times:')
print('  TEMPORAL : ~5 min (coarse resolution)')
print('  DEM      : ~30-60 min')
print('  S1/S2    : ~1-3 hours each (10m over Assam)')

DEM export started ✅  ID: CBSUB4ND7G33FP4RFUGATT2K
S1 flood export started ✅  ID: 6WFLQPVW2NUE26WAALIOTQMF
S1 pre-flood export started ✅  ID: ASUHB24TDXJI6FSBWRCFFCYY
S2 export started ✅  ID: ZIJV6WQADXAHH6N6DTYMLP2D
Temporal export started ✅  ID: S7ZDOAYWZ6A4PSUZEZY75ZMR

All 5 exports running in parallel ✅
Monitor: https://code.earthengine.google.com/tasks

Expected times:
  TEMPORAL : ~5 min (coarse resolution)
  DEM      : ~30-60 min
  S1/S2    : ~1-3 hours each (10m over Assam)


In [ ]:
import os, rasterio
import numpy as np
from osgeo import gdal

DRIVE_BASE = '/content/drive/MyDrive/FloodProject_Assam2023'

# 1. List all files
all_tifs = sorted([f for f in os.listdir(DRIVE_BASE) if f.endswith('.tif')])
print(f'Total tifs: {len(all_tifs)}')
for f in all_tifs:
    size_mb = os.path.getsize(f'{DRIVE_BASE}/{f}') / 1e6
    print(f'  {f:<60} {size_mb:.1f} MB')

Total tifs: 47
  Assam2023_TEMPORAL.tif                                       0.1 MB
  assam2023_DEM-0000000000-0000000000.tif                      456.5 MB
  assam2023_DEM-0000000000-0000016384.tif                      587.4 MB
  assam2023_DEM-0000000000-0000032768.tif                      757.8 MB
  assam2023_DEM-0000000000-0000049152.tif                      1304.7 MB
  assam2023_DEM-0000000000-0000065536.tif                      229.9 MB
  assam2023_DEM-0000016384-0000000000.tif                      910.9 MB
  assam2023_DEM-0000016384-0000016384.tif                      1064.5 MB
  assam2023_DEM-0000016384-0000032768.tif                      1848.0 MB
  assam2023_DEM-0000016384-0000049152.tif                      33.2 MB
  assam2023_DEM-0000016384-0000065536.tif                      5.1 MB
  assam2023_DEM-0000032768-0000000000.tif                      10.2 MB
  assam2023_DEM-0000032768-0000016384.tif                      364.8 MB
  assam2023_DEM-0000032768-0000032768.tif           

In [ ]:
import rasterio
import numpy as np

DRIVE_BASE = '/content/drive/MyDrive/FloodProject_Assam2023'

sources = {
    'S1_flood' : 'assam2023_S1_flood-0000000000-0000000000.tif',
    'S1_pre'   : 'assam2023_S1_pre-0000000000-0000000000.tif',
    'S2'       : 'assam2023_S2-0000000000-0000000000.tif',
    'DEM'      : 'assam2023_DEM-0000000000-0000000000.tif',
    'TEMPORAL' : 'Assam2023_TEMPORAL.tif',
    'Label'    : 'assam_flood_label_s1s2_aug2023-0000000000-0000000000.tif',
}

for name, fname in sources.items():
    with rasterio.open(f'{DRIVE_BASE}/{fname}') as src:
        # Read only a tiny 64x64 window — no RAM pressure
        win = rasterio.windows.Window(0, 0, 64, 64)
        arr = src.read(window=win)
        print(f'{name:<10} bands={src.count}  dtype={src.dtypes[0]}  '
              f'shape={src.height}×{src.width}  '
              f'res={src.res[0]:.6f}')
        del arr

S1_flood   bands=2  dtype=float32  shape=23296×23296  res=0.000090
S1_pre     bands=2  dtype=float32  shape=23296×23296  res=0.000090
S2         bands=6  dtype=int16  shape=18944×18944  res=0.000090
DEM        bands=4  dtype=float32  shape=16384×16384  res=0.000090
TEMPORAL   bands=30  dtype=float32  shape=44×71  res=0.089832
Label      bands=1  dtype=int8  shape=42777×65536  res=0.000090


# PART B — Model Inference + IoU Evaluation

**Prerequisite:** GEE exports from Part A must be complete and present in Drive.

**Models:**
- Variant 1: U-Net spatial only (baseline, IoU=0.7360 on Sen1Floods11 test)
- Variant 5: U-Net + Terrain-Conditioned Temporal MLP (best, IoU=0.7367)

**Pipeline:** VRT mosaic → chip-by-chip inference (512×512, stride=480) → stitch prediction → compute IoU vs reference mask

In [ ]:
import gc
import torch
import os

# Limit CPU threads to reduce memory overhead
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'

# Check available RAM
import psutil
ram = psutil.virtual_memory()
print(f'Available RAM: {ram.available/1e9:.1f} GB / {ram.total/1e9:.1f} GB')
print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Available RAM: 11.7 GB / 13.6 GB
GPU memory: 15.6 GB


In [ ]:
# ── Install / import dependencies ─────────────────────────────────────────
!pip install -q rasterio
!apt-get install -q gdal-bin python3-gdal

import numpy as np
import rasterio
from rasterio.windows import Window
from osgeo import gdal
import torch
import torch.nn as nn
import torch.nn.functional as F
import json, gc, os, warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Reading package lists...
Building dependency tree...
Reading state information...
gdal-bin is already the newest version (3.8.4+dfsg-1~jammy0).
python3-gdal is already the newest version (3.8.4+dfsg-1~jammy0).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.
Device: cuda


In [ ]:
import os

# ── Paths ─────────────────────────────────────────────────────────────────
DRIVE_BASE    = '/content/drive/MyDrive/FloodProject_Assam2023'
MODEL_BASE    = '/content/drive/MyDrive/FloodProject/models'
LOCAL_BASE    = '/content/assam2023'

# Model checkpoints — update these paths to match your Drive structure
# Kaggle saves models to /kaggle/working/models/ — copy them to Drive
VARIANT1_CKPT = f'{MODEL_BASE}/unet_spatial/best_model.pth'
VARIANT5_CKPT = f'{MODEL_BASE}/unet_temporal_mlp/best_model.pth'

# Norm stats (same file used during training)
NORM_STATS    = '/content/drive/MyDrive/FloodProject/data/preprocessed/norm_stats.json'

os.makedirs(LOCAL_BASE, exist_ok=True)
print('Paths set ✅')

Paths set ✅


In [ ]:
# ── U-Net architecture (Variant 1) ────────────────────────────────────────
# Identical to training — self.out (NOT self.head) to match saved checkpoint

# ── Shared building blocks ────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = ConvBlock(in_ch, out_ch)
    def forward(self, x): return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = ConvBlock(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        return self.conv(torch.cat([self.up(x), skip], dim=1))

# ── Variant 1 ─────────────────────────────────────────────────
class UNet(nn.Module):
    def __init__(self, in_channels=14):
        super().__init__()
        f = 64
        self.enc1       = ConvBlock(in_channels, f)
        self.enc2       = Down(f,    f*2)
        self.enc3       = Down(f*2,  f*4)
        self.enc4       = Down(f*4,  f*8)
        self.bottleneck = Down(f*8,  f*16)
        self.dec4       = Up(f*16, f*8,  f*8)
        self.dec3       = Up(f*8,  f*4,  f*4)
        self.dec2       = Up(f*4,  f*2,  f*2)
        self.dec1       = Up(f*2,  f,    f)
        self.out        = nn.Conv2d(f, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bottleneck(e4)
        d4 = self.dec4(b,  e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.out(d1)

# ── Variant 5: U-Net + Terrain-Conditioned Temporal MLP ──────
# ── U-Net + Terrain-Conditioned Temporal MLP (Variant 5) ──────────────────
# Temporal features (5 scalars per chip):
#   0: CHIRPS 3-day sum  (immediate rainfall trigger)
#   1: CHIRPS 7-day sum  (short antecedent wetting)
#   2: CHIRPS 15-day sum (long antecedent wetting)
#   3: ERA5 mean         (saturation state)
#   4: ERA5 trend        (is soil getting wetter or drier?)
#
# The MLP output is terrain-gated via HAND: low-lying areas respond more to
# antecedent rainfall than hillsides.
HAND_CHANNEL = 11  # Elev(8),Slope(9),TWI(10),HAND(11)

class TerrainConditionedTemporalMLP(nn.Module):
    def __init__(self, n_features, hidden_size, out_channels, bottleneck_size=32):
        super().__init__()
        self.bottleneck_size = bottleneck_size
        self.mlp = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_size, out_channels),
        )
        # Sigmoid is INSIDE hand_gate Sequential
        self.hand_gate = nn.Sequential(
            nn.Conv2d(1, out_channels, 1, bias=True),
            nn.Sigmoid(),
        )
        self.hand_pool = nn.AdaptiveAvgPool2d((bottleneck_size, bottleneck_size))

    def forward(self, temp_feats, spatial):
        t    = self.mlp(temp_feats)
        t    = t.unsqueeze(-1).unsqueeze(-1)
        t    = t.expand(-1, -1, self.bottleneck_size, self.bottleneck_size)
        hand = spatial[:, HAND_CHANNEL:HAND_CHANNEL+1, :, :]
        hand = self.hand_pool(hand)
        gate = self.hand_gate(hand)
        return t * gate

class UNetTemporalMLP(nn.Module):
    def __init__(self, spatial_channels=14, n_temporal_feats=5,
                 mlp_hidden=128, base_features=64):
        super().__init__()
        f = base_features

        self.enc1    = ConvBlock(spatial_channels, f)
        self.enc2    = Down(f,   f*2)
        self.enc3    = Down(f*2, f*4)
        self.enc4    = Down(f*4, f*8)
        self.down_bn = nn.MaxPool2d(2)

        self.temporal_mlp = TerrainConditionedTemporalMLP(
            n_features      = n_temporal_feats,
            hidden_size     = mlp_hidden,
            out_channels    = f*8,
            bottleneck_size = 32,
        )

        # fusion: (f*8 + f*8) → f*8, with BN + ReLU
        self.fusion_conv = nn.Sequential(
            nn.Conv2d(f*8 + f*8, f*8, 1, bias=False),
            nn.BatchNorm2d(f*8),
            nn.ReLU(inplace=True),
        )

        # bottleneck comes AFTER fusion
        self.bottleneck = ConvBlock(f*8, f*16)

        self.dec4 = Up(f*16, f*8,  f*8)
        self.dec3 = Up(f*8,  f*4,  f*4)
        self.dec2 = Up(f*4,  f*2,  f*2)
        self.dec1 = Up(f*2,  f,    f)
        self.head = nn.Conv2d(f, 1, 1)

    def forward(self, spatial, temp_feats):
        e1   = self.enc1(spatial)
        e2   = self.enc2(e1)
        e3   = self.enc3(e2)
        e4   = self.enc4(e3)
        s_bn = self.down_bn(e4)

        t_feat = self.temporal_mlp(temp_feats, spatial)

        fused = torch.cat([s_bn, t_feat], dim=1)
        fused = self.fusion_conv(fused)
        bn    = self.bottleneck(fused)

        d4 = self.dec4(bn, e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.head(d1)

print('Model classes defined ✅')

Model classes defined ✅


In [ ]:
# ── Load model checkpoints ─────────────────────────────────────────────────
model_v1 = UNet(in_channels=14).to(DEVICE)
model_v1.load_state_dict(torch.load(VARIANT1_CKPT, map_location=DEVICE))
model_v1.eval()
print('Variant 1 loaded ✅')

model_v5 = UNetTemporalMLP().to(DEVICE)
model_v5.load_state_dict(torch.load(VARIANT5_CKPT, map_location=DEVICE))
model_v5.eval()
print('Variant 5 loaded ✅')

Variant 1 loaded ✅
Variant 5 loaded ✅


In [ ]:
# Build VRTs
def build_vrt(pattern, out_vrt):
    tiles = sorted([
        f'{DRIVE_BASE}/{f}' for f in os.listdir(DRIVE_BASE)
        if pattern in f and f.endswith('.tif') and '(1)' not in f
    ])
    print(f'{pattern}: {len(tiles)} tiles')
    gdal.BuildVRT(out_vrt, tiles)
    with rasterio.open(out_vrt) as src:
        print(f'  → {src.height}×{src.width}, {src.count} bands, {src.crs}')
    return out_vrt

vrt_s1f   = build_vrt('S1_flood',   f'{LOCAL_BASE}/S1_flood.vrt')
vrt_s1p   = build_vrt('S1_pre',     f'{LOCAL_BASE}/S1_pre.vrt')
vrt_s2    = build_vrt('assam2023_S2',  f'{LOCAL_BASE}/S2.vrt')
vrt_dem   = build_vrt('assam2023_DEM', f'{LOCAL_BASE}/DEM.vrt')
vrt_label = build_vrt('flood_label',   f'{LOCAL_BASE}/label.vrt')

# Get mosaic reference dimensions from S1
with rasterio.open(vrt_s1f) as src:
    MOSAIC_H  = src.height
    MOSAIC_W  = src.width
    meta      = src.meta.copy()
    transform = src.transform
    crs       = src.crs

print(f'\nMosaic reference: {MOSAIC_H}×{MOSAIC_W}')

S1_flood: 8 tiles
  → 42777×70422, 2 bands, EPSG:4326
S1_pre: 8 tiles
  → 42777×70422, 2 bands, EPSG:4326
assam2023_S2: 12 tiles
  → 42777×70422, 6 bands, EPSG:4326
assam2023_DEM: 15 tiles
  → 42777×70422, 4 bands, EPSG:4326
flood_label: 2 tiles
  → 42777×70422, 1 bands, EPSG:4326

Mosaic reference: 42777×70422


In [ ]:
# ── Cell 7: Norm Stats + Temporal Features ────────────────────
with open(NORM_STATS) as f:
    stats = json.load(f)

sp_mean = np.array(stats['spatial']['mean'], dtype=np.float32)   # (14,)
sp_std  = np.array(stats['spatial']['std'],  dtype=np.float32)   # (14,)
tp_mean = np.array(stats['temporal']['mean'], dtype=np.float32)  # (2,)
tp_std  = np.array(stats['temporal']['std'],  dtype=np.float32)
print(f'Norm stats loaded ✅  spatial={sp_mean.shape}  temporal={tp_mean.shape}')

# Load temporal — tiny file (44×71), safe to load fully
with rasterio.open(f'{DRIVE_BASE}/Assam2023_TEMPORAL.tif') as src:
    temp_data = src.read().astype(np.float32)  # (30, 44, 71)

# Band order: chirps_day00, era5_day00, chirps_day01, era5_day01, ...
chirps = np.nanmean(temp_data[0::2], axis=(-2,-1))  # (15,) — CHIRPS days
era5   = np.nanmean(temp_data[1::2], axis=(-2,-1))  # (15,) — ERA5 days

chirps_norm = (chirps - tp_mean[0]) / (tp_std[0] + 1e-8)
era5_norm   = (era5   - tp_mean[1]) / (tp_std[1] + 1e-8)

# 5 temporal features — same as training
temp_feats = np.array([
    chirps_norm[-3:].sum(),                          # 3-day accumulated rain
    chirps_norm[-7:].sum(),                          # 7-day accumulated rain
    chirps_norm.sum(),                               # 15-day accumulated rain
    era5_norm.mean(),                                # mean soil moisture
    np.polyfit(np.arange(15), era5_norm, 1)[0],     # soil moisture trend
], dtype=np.float32)

temp_tensor = torch.tensor(temp_feats).unsqueeze(0).to(DEVICE)  # (1, 5)
del temp_data
gc.collect()

print(f'Temporal features: {temp_feats}')
print('  [3d-rain, 7d-rain, 15d-rain, era5-mean, era5-trend]')

Norm stats loaded ✅  spatial=(14,)  temporal=(2,)
Temporal features: [1.8171532e+00 2.1850243e+00 3.8922355e+00 4.7714123e-01 2.8215623e-03]
  [3d-rain, 7d-rain, 15d-rain, era5-mean, era5-trend]


In [ ]:
# ── Cell 8: Chip Reader ───────────────────────────────────────
# Channel order: VV(0),VH(1),B2(2),B3(3),B4(4),B8(5),B11(6),B12(7),
#                Elev(8),Slope(9),TWI(10),HAND(11),NDWI(12),NDVI(13)
CHIP_SIZE = 512
STRIDE    = 512
THRESHOLD = 0.5

def read_chip(vrt_path, row, col):
    win = Window(col, row, CHIP_SIZE, CHIP_SIZE)
    with rasterio.open(vrt_path) as src:
        return src.read(window=win, boundless=True,
                        fill_value=np.nan).astype(np.float32)

def make_spatial_tensor(row, col):
    s1f = read_chip(vrt_s1f, row, col)   # (2,512,512) VV,VH flood
    s2  = read_chip(vrt_s2,  row, col)   # (6,512,512) B2,B3,B4,B8,B11,B12
    dem = read_chip(vrt_dem, row, col)   # (4,512,512) Elev,Slope,TWI,HAND

    if np.isnan(s1f[0]).mean() > 0.9:
        return None

    b3 = s2[1].astype(np.float32)
    b4 = s2[2].astype(np.float32)
    b8 = s2[3].astype(np.float32)
    ndwi = np.clip((b3 - b8) / (b3 + b8 + 1e-8), -1, 1)[None]
    ndvi = np.clip((b8 - b4) / (b8 + b4 + 1e-8), -1, 1)[None]

    # 14 channels: VV(0),VH(1),B2(2),B3(3),B4(4),B8(5),
    #              B11(6),B12(7),Elev(8),Slope(9),TWI(10),
    #              HAND(11),NDWI(12),NDVI(13)
    chip = np.concatenate([s1f, s2, dem, ndwi, ndvi], axis=0)

    for ch_idx in range(chip.shape[0]):
        chip[ch_idx][np.isnan(chip[ch_idx])] = sp_mean[ch_idx]

    chip = (chip - sp_mean[:,None,None]) / (sp_std[:,None,None] + 1e-8)
    return chip.astype(np.float32)

print('Chip reader defined ✅')

Chip reader defined ✅


In [ ]:
# ── Cell 9: Create Output GeoTIFFs ────────────────────────────
def create_output_tif(path):
    meta_out = meta.copy()
    meta_out.update(count=1, dtype='uint8', compress='lzw',
                    driver='GTiff', bigtiff='YES')
    with rasterio.open(path, 'w', **meta_out) as dst:
        pass
    return path

out_v1 = create_output_tif(f'{DRIVE_BASE}/assam_pred_v1.tif')
out_v5 = create_output_tif(f'{DRIVE_BASE}/assam_pred_v5.tif')
print(f'Output files created ✅')
print(f'  {out_v1}')
print(f'  {out_v5}')

Output files created ✅
  /content/drive/MyDrive/FloodProject_Assam2023/assam_pred_v1.tif
  /content/drive/MyDrive/FloodProject_Assam2023/assam_pred_v5.tif


In [ ]:
# Cell 10: Inference Loop (RESUME VERSION)
# Skips chips where out_v1 already has data written
processed = skipped = resumed = 0

with rasterio.open(out_v1, 'r+') as dst_v1, \
     rasterio.open(out_v5, 'r+') as dst_v5, \
     torch.no_grad():

    rows = list(range(0, MOSAIC_H, STRIDE))
    cols = list(range(0, MOSAIC_W, STRIDE))
    pbar = tqdm(total=len(rows)*len(cols), desc='Inference')

    for r in rows:
        for c in cols:
            pbar.update(1)

            h_out = min(CHIP_SIZE, MOSAIC_H - r)
            w_out = min(CHIP_SIZE, MOSAIC_W - c)
            win   = Window(c, r, w_out, h_out)

            # Resume: check if this chip was already written
            # Any non-zero value in the first pixel means it was processed
            existing = dst_v1.read(1, window=Window(c, r, 1, 1))
            if existing[0, 0] != 0:
                resumed += 1
                continue

            chip_np = make_spatial_tensor(r, c)
            if chip_np is None:
                skipped += 1
                # Write a 1 sentinel so we don't re-check this outside-AOI chip
                dst_v1.write(np.array([[[1]]], dtype=np.uint8), window=Window(c, r, 1, 1))
                dst_v5.write(np.array([[[1]]], dtype=np.uint8), window=Window(c, r, 1, 1))
                continue

            chip_t = torch.tensor(chip_np).unsqueeze(0).to(DEVICE)

            prob_v1 = torch.sigmoid(model_v1(chip_t)).squeeze().cpu().numpy()
            pred_v1 = (prob_v1[:h_out, :w_out] >= THRESHOLD).astype(np.uint8)
            pred_v1[0, 0] = max(pred_v1[0, 0], 1)  # ensure non-zero sentinel
            dst_v1.write(pred_v1[None], window=win)

            prob_v5 = torch.sigmoid(model_v5(chip_t, temp_tensor)).squeeze().cpu().numpy()
            pred_v5 = (prob_v5[:h_out, :w_out] >= THRESHOLD).astype(np.uint8)
            pred_v5[0, 0] = max(pred_v5[0, 0], 1)
            dst_v5.write(pred_v5[None], window=win)

            del chip_t, prob_v1, prob_v5, pred_v1, pred_v5, chip_np
            processed += 1

            if processed % 200 == 0:
                gc.collect()
                torch.cuda.empty_cache()
                pbar.set_postfix({'done': processed, 'skip': skipped, 'resume': resumed})

    pbar.close()

print(f'\nDone ✅  processed={processed}  skipped={skipped}  resumed={resumed}')

Inference:  69%|██████▊   | 7943/11592 [1:54:53<2:14:30,  2.21s/it]

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── Cell 11: IoU Evaluation ───────────────────────────────────
# Chip-by-chip to avoid loading full label into RAM

tp_v1 = fp_v1 = fn_v1 = 0
tp_v5 = fp_v5 = fn_v5 = 0

with rasterio.open(vrt_label) as src_lbl, \
     rasterio.open(out_v1)    as src_v1,  \
     rasterio.open(out_v5)    as src_v5:

    rows = list(range(0, MOSAIC_H, STRIDE))
    cols = list(range(0, MOSAIC_W, STRIDE))
    pbar = tqdm(total=len(rows)*len(cols), desc='IoU')

    for r in rows:
        for c in cols:
            pbar.update(1)
            h_out = min(CHIP_SIZE, MOSAIC_H - r)
            w_out = min(CHIP_SIZE, MOSAIC_W - c)
            win   = Window(c, r, w_out, h_out)

            lbl  = src_lbl.read(1, window=win, boundless=True, fill_value=-1)
            pred1 = src_v1.read(1, window=win, boundless=True, fill_value=0)
            pred5 = src_v5.read(1, window=win, boundless=True, fill_value=0)

            # Only evaluate valid pixels (label != -1)
            valid = lbl != -1
            if valid.sum() == 0:
                continue

            g  = lbl[valid] == 1
            p1 = pred1[valid].astype(bool)
            p5 = pred5[valid].astype(bool)

            tp_v1 += int((p1 & g).sum())
            fp_v1 += int((p1 & ~g).sum())
            fn_v1 += int((~p1 & g).sum())

            tp_v5 += int((p5 & g).sum())
            fp_v5 += int((p5 & ~g).sum())
            fn_v5 += int((~p5 & g).sum())

    pbar.close()


def metrics(tp, fp, fn):
    iou  = tp / (tp + fp + fn + 1e-8)
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return iou, prec, rec, f1

iou_v1, prec_v1, rec_v1, f1_v1 = metrics(tp_v1, fp_v1, fn_v1)
iou_v5, prec_v5, rec_v5, f1_v5 = metrics(tp_v5, fp_v5, fn_v5)

print()
print('=' * 62)
print('  ASSAM 2023 GENERALIZATION TEST RESULTS')
print('=' * 62)
print(f'  {"Model":<30} {"IoU":>6} {"Prec":>6} {"Rec":>6} {"F1":>6}')
print(f'  {"-"*58}')
print(f'  {"Variant 1 (U-Net spatial)":<30} {iou_v1:.4f} {prec_v1:.4f} {rec_v1:.4f} {f1_v1:.4f}')
print(f'  {"Variant 5 (U-Net+TempMLP)":<30} {iou_v5:.4f} {prec_v5:.4f} {rec_v5:.4f} {f1_v5:.4f}')
print('=' * 62)
print(f'  Sen1Floods11 test ref : V1=0.7360  V5=0.7367')
print('=' * 62)

NameError: name 'rasterio' is not defined

In [ ]:
# ── Save prediction rasters to Drive ──────────────────────────────────────
# Useful for visual inspection in QGIS

def save_prediction(arr, filename, dtype=rasterio.uint8):
    out_path = f'{DRIVE_BASE}/{filename}'
    with rasterio.open(vrt_s1_flood) as src:
        meta = src.meta.copy()
    meta.update(count=1, dtype=dtype, compress='lzw', driver='GTiff')
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(arr.astype(dtype), 1)
    print(f'Saved: {out_path}')

# Save binary predictions
save_prediction(pred_bin_v1.astype(np.uint8), 'assam_pred_variant1_unet.tif')
save_prediction(pred_bin_v5.astype(np.uint8), 'assam_pred_variant5_temporal_mlp.tif')

# Save probability maps (scaled to 0-255 for storage)
save_prediction((pred_avg_v1 * 255).astype(np.uint8), 'assam_prob_variant1_unet.tif')
save_prediction((pred_avg_v5 * 255).astype(np.uint8), 'assam_prob_variant5_temporal_mlp.tif')

print('All predictions saved to Drive ✅')

In [ ]:
# ── Quick visual comparison ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Sample a ~512×512 region with known flooding for visualization
# Adjust r0, c0 to a flood-heavy area (check QGIS or the label map)
r0, c0 = 8000, 20000  # example — central Assam / Brahmaputra floodplain
sz     = 512

label_crop  = label_int[r0:r0+sz, c0:c0+sz]
pred_v1_crop = pred_bin_v1[r0:r0+sz, c0:c0+sz]
pred_v5_crop = pred_bin_v5[r0:r0+sz, c0:c0+sz]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cmap = plt.cm.get_cmap('RdYlBu')

axes[0].imshow(label_crop, cmap='Blues', vmin=-1, vmax=1)
axes[0].set_title('Reference Label\n(blue=flood, grey=invalid)', fontsize=11)

axes[1].imshow(pred_v1_crop, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title(f'Variant 1 (U-Net)\nIoU={iou_v1:.4f}', fontsize=11)

axes[2].imshow(pred_v5_crop, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title(f'Variant 5 (U-Net + Temporal MLP)\nIoU={iou_v5:.4f}', fontsize=11)

for ax in axes:
    ax.axis('off')

plt.suptitle('Assam 2023 Flood — Generalization Test', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/assam_generalization_visual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualization saved ✅')

In [ ]:
# ── GEE Setup (same as before) ────────────────────────────────
import ee
ee.Authenticate()
ee.Initialize(project='Replace with your own GEE project ID')

from google.colab import drive
drive.mount('/content/drive')

states = ee.FeatureCollection('FAO/GAUL/2015/level1')
assam  = states.filter(ee.Filter.eq('ADM1_NAME', 'Assam'))
aoi    = assam.geometry()

# ── Rebuild S2 + S1 (reuse from before — already in Drive) ────
def mask_s2_clouds(img):
    scl   = img.select('SCL')
    clear = (scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(11)))
    return img.updateMask(clear).divide(10000).copyProperties(img, ['system:time_start'])

s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate('2023-08-19', '2023-09-08')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
    .map(mask_s2_clouds))

s2_median = s2_col.median().clip(aoi)
mndwi     = s2_median.normalizedDifference(['B3', 'B11']).rename('MNDWI')
s2_valid  = s2_col.select('B3').count().gt(0).rename('s2_valid').clip(aoi)

s1_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2023-08-23', '2023-09-05')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .select(['VH', 'VV']))

def speckle_filter(img):
    vh = img.select('VH').focal_mean(radius=50, kernelType='circle', units='meters')
    return img.addBands(vh.rename('VH_sm'))

s1_filtered = s1_col.map(speckle_filter)
s1_img      = s1_filtered.median().clip(aoi)
s1_water    = s1_img.select('VH_sm').lt(-16).rename('s1_water')
s1_valid    = s1_col.select('VH').count().gt(0).rename('s1_valid').clip(aoi)

jrc        = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
perm_water = jrc.select('seasonality').gte(10)
any_valid  = s2_valid.rename('valid').Or(s1_valid.rename('valid'))

print('Data sources rebuilt ✅')

# ── Test 4 thresholds — check flood fraction before exporting ──
print('\nTesting MNDWI thresholds:')
print(f'{"Threshold":>12}  {"Flood fraction":>15}  {"Flood area (km²)":>18}')
print('-' * 50)

for thresh in [0.0, -0.05, -0.1, -0.15, -0.2]:
    s2_water_t     = mndwi.gt(thresh).rename('s2_water')
    fused_t        = s2_water_t.unmask(0).where(s2_valid.eq(0), s1_water)
    flood_only_t   = fused_t.where(perm_water, ee.Image(0))
    label_t        = flood_only_t.multiply(any_valid.rename('valid').gt(0)).toInt8().clip(aoi)

    stats = label_t.eq(1).reduceRegion(
        reducer   = ee.Reducer.mean(),
        geometry  = aoi,
        scale     = 1000,
        maxPixels = 1e10
    ).getInfo()

    frac     = stats.get(list(stats.keys())[0], 0)
    area_km2 = frac * 78438  # Assam area in km²
    marker   = ' ← current' if thresh == 0.0 else ''
    print(f'{thresh:>12.2f}  {frac:>15.4f}  {area_km2:>16.0f} km²{marker}')

print()
print('Physical GEE estimate: ~9.15% (~7,400 km²)')
print('Pick the threshold closest to 9.15%')

Mounted at /content/drive
Data sources rebuilt ✅

Testing MNDWI thresholds:
   Threshold   Flood fraction    Flood area (km²)
--------------------------------------------------
        0.00           0.0915              7176 km² ← current
       -0.05           0.1017              7978 km²
       -0.10           0.1175              9218 km²
       -0.15           0.1412             11076 km²
       -0.20           0.1726             13540 km²

Physical GEE estimate: ~9.15% (~7,400 km²)
Pick the threshold closest to 9.15%


In [ ]:
# ── SAR-based reference label ─────────────────────────────────
# Method: S1 VH change detection (flood vs pre-flood)
# Flood pixel: VH_flood significantly lower than VH_pre
# + absolute VH threshold for open water
# + JRC permanent water removal
# Matches Sen1Floods11 label generation methodology

# Pre-flood S1 (Jan-May 2023 dry season baseline)
s1_pre_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2023-01-01', '2023-05-31')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .select(['VH']))

s1_pre_img = s1_pre_col.map(
    lambda img: img.focal_mean(radius=50, kernelType='circle', units='meters')
).median().clip(aoi).rename('VH_pre')

# Flood S1 (Aug 2023)
s1_flood_img = s1_img.select('VH_sm').rename('VH_flood')

# Change detection: flood - pre (in dB, so negative = darker = more water)
vh_change = s1_flood_img.subtract(s1_pre_img).rename('VH_change')

# Three-part flood mask (matching Sen1Floods11 approach):

# Part 1: Open water — absolute VH threshold
# VH < -16 dB = open water (Brahmaputra, lakes, flooded fields)
open_water = s1_flood_img.lt(-16).rename('open_water')

# Part 2: Flooded vegetation — change detection
# VH dropped by more than 3 dB = significant inundation change
# (flooded rice paddies, wetlands — this is what MNDWI misses)
flooded_veg = vh_change.lt(-3).rename('flooded_veg')

# Part 3: Combine — either open water OR significant change
sar_flood = open_water.Or(flooded_veg).rename('sar_flood')

# Remove permanent water (JRC)
sar_flood_only = sar_flood.where(perm_water, ee.Image(0)).rename('sar_flood_only')

# Remove steep slopes (false positives from layover/shadow)
dem      = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic()
slope    = ee.Terrain.slope(dem)
flat     = slope.lt(5)  # keep only areas with <5° slope
sar_flood_only = sar_flood_only.where(flat.Not(), ee.Image(0))

# Final label: 1=flood, 0=land (no -1 since SAR has full coverage)
sar_label = sar_flood_only.toInt8().clip(aoi)

# ── Check flood fractions ──────────────────────────────────────
print('Checking SAR label flood fractions...')

# Open water only
ow_stats = open_water.where(perm_water,0).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e10
).getInfo()

# Change detection only
cd_stats = flooded_veg.where(perm_water,0).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e10
).getInfo()

# Combined
sar_stats = sar_label.eq(1).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e10
).getInfo()

ow_frac  = list(ow_stats.values())[0]
cd_frac  = list(cd_stats.values())[0]
sar_frac = list(sar_stats.values())[0]

print(f'Open water only     : {ow_frac:.4f} ({ow_frac*78438:.0f} km²)')
print(f'Flooded vegetation  : {cd_frac:.4f} ({cd_frac*78438:.0f} km²)')
print(f'Combined SAR label  : {sar_frac:.4f} ({sar_frac*78438:.0f} km²)')
print(f'MNDWI label (current): 0.0915 (7176 km²)')
print(f'Physical estimate   : ~0.0915 (~7400 km²)')
print()
print('Target: combined SAR label ~10-15% (captures flooded vegetation)')

Checking SAR label flood fractions...
Open water only     : 0.2596 (20366 km²)
Flooded vegetation  : 0.0354 (2776 km²)
Combined SAR label  : 0.2598 (20376 km²)
MNDWI label (current): 0.0915 (7176 km²)
Physical estimate   : ~0.0915 (~7400 km²)

Target: combined SAR label ~10-15% (captures flooded vegetation)


In [ ]:
# Tune open water threshold
print('Tuning open water VH threshold:')
print(f'{"VH threshold":>14}  {"Flood fraction":>15}  {"Area (km²)":>12}')
print('-' * 45)

for thresh in [-16, -17, -18, -19, -20, -22, -24]:
    ow   = s1_flood_img.lt(thresh)
    comb = ow.Or(flooded_veg).where(perm_water, 0).where(flat.Not(), 0)
    stats = comb.eq(1).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e10
    ).getInfo()
    frac = list(stats.values())[0]
    area = frac * 78438
    marker = ' ← target range' if 0.09 <= frac <= 0.15 else ''
    print(f'{thresh:>14}  {frac:>15.4f}  {area:>12.0f} km²{marker}')

print()
print('MNDWI label: 0.0915 (7176 km²)')
print('Physical estimate: ~0.0915 (~7400 km²)')
print('Target: 10-15% to capture flooded vegetation the MNDWI missed')

Tuning open water VH threshold:
  VH threshold   Flood fraction    Area (km²)
---------------------------------------------
           -16           0.2598         20376 km²
           -17           0.1565         12274 km²
           -18           0.1058          8298 km² ← target range
           -19           0.0794          6226 km²
           -20           0.0675          5296 km²
           -22           0.0566          4439 km²
           -24           0.0503          3948 km²

MNDWI label: 0.0915 (7176 km²)
Physical estimate: ~0.0915 (~7400 km²)
Target: 10-15% to capture flooded vegetation the MNDWI missed


In [ ]:
# ── Final SAR label with tuned threshold ──────────────────────
open_water_final  = s1_flood_img.lt(-18)
flooded_veg_final = vh_change.lt(-3)
sar_flood_final   = open_water_final.Or(flooded_veg_final)
sar_label_final   = (sar_flood_final
    .where(perm_water, ee.Image(0))   # remove permanent water
    .where(flat.Not(), ee.Image(0))   # remove steep slopes
    .toInt8()
    .clip(aoi))

# Final check
stats = sar_label_final.eq(1).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e10
).getInfo()
frac = list(stats.values())[0]
print(f'Final SAR label flood fraction: {frac:.4f} ({frac*78438:.0f} km²) ✅')

# ── Export ────────────────────────────────────────────────────
task = ee.batch.Export.image.toDrive(
    image          = sar_label_final,
    description    = 'Assam_SAR_label_final',
    folder         = 'FloodProject_Assam2023',
    fileNamePrefix = 'assam_sar_label_final',
    region         = aoi,
    scale          = 10,
    crs            = 'EPSG:4326',
    maxPixels      = 1e11,
    fileFormat     = 'GeoTIFF'
)
task.start()
print(f'SAR label export started ✅  ID: {task.id}')
print('Monitor: https://code.earthengine.google.com/tasks')
print('Expected time: 45-90 mins')

Final SAR label flood fraction: 0.1058 (8298 km²) ✅
SAR label export started ✅  ID: 6F76UVJH25NYYVL74BIEDFCX
Monitor: https://code.earthengine.google.com/tasks
Expected time: 45-90 mins


In [ ]:
# Export VH change map (useful figure for paper)
task_change = ee.batch.Export.image.toDrive(
    image          = vh_change.multiply(100).toInt16().clip(aoi),
    description    = 'Assam_VH_change_map',
    folder         = 'FloodProject_Assam2023',
    fileNamePrefix = 'assam_vh_change_map',
    region         = aoi,
    scale          = 30,   # coarser ok for diagnostic
    crs            = 'EPSG:4326',
    maxPixels      = 1e11,
    fileFormat     = 'GeoTIFF'
)
task_change.start()
print(f'VH change map export started ✅  ID: {task_change.id}')

VH change map export started ✅  ID: 2LMYOCJUIQSSYQTGOOMCKDSF


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, rasterio
import numpy as np

DRIVE_BASE = '/content/drive/MyDrive/FloodProject_Assam2023'

# Find SAR label tiles
sar_tiles = [f for f in os.listdir(DRIVE_BASE) if 'sar_label' in f and f.endswith('.tif')]
print(f'SAR label tiles: {len(sar_tiles)}')
for f in sorted(sar_tiles):
    size = os.path.getsize(f'{DRIVE_BASE}/{f}') / 1e6
    print(f'  {f}  ({size:.1f} MB)')

# Check first tile
tile = f'{DRIVE_BASE}/{sorted(sar_tiles)[0]}'
with rasterio.open(tile) as src:
    chip = src.read(1, window=rasterio.windows.Window(0, 0, 64, 64))
    vals, counts = np.unique(chip, return_counts=True)
    print(f'\nCRS: {src.crs}')
    print(f'Shape: {src.height}×{src.width}')
    print(f'Dtype: {src.dtypes[0]}')
    print(f'Values: {dict(zip(vals, counts))}')

SAR label tiles: 2
  assam_sar_label_final-0000000000-0000000000.tif  (36.5 MB)
  assam_sar_label_final-0000000000-0000065536.tif  (1.8 MB)

CRS: EPSG:4326
Shape: 42777×65536
Dtype: int8
Values: {np.int8(0): np.int64(4096)}


In [ ]:
import os
from osgeo import gdal
import rasterio
import numpy as np

DRIVE_BASE = '/content/drive/MyDrive/FloodProject_Assam2023'
LOCAL_BASE = '/content/assam2023'
os.makedirs(LOCAL_BASE, exist_ok=True)  # ← create it first

# Build VRT
sar_tiles = sorted([f'{DRIVE_BASE}/{f}' for f in os.listdir(DRIVE_BASE)
                    if 'sar_label' in f and f.endswith('.tif')])
print(f'Found {len(sar_tiles)} SAR label tiles')
gdal.BuildVRT(f'{LOCAL_BASE}/sar_label.vrt', sar_tiles)

with rasterio.open(f'{LOCAL_BASE}/sar_label.vrt') as src:
    data = src.read(1, out_shape=(src.height//100, src.width//100))
    vals, counts = np.unique(data, return_counts=True)
    total = counts.sum()
    for v, c in zip(vals, counts):
        print(f'Value {v}: {c:,} pixels ({c/total*100:.2f}%)')

Found 2 SAR label tiles
Value 0: 280,541 pixels (93.32%)
Value 1: 20,067 pixels (6.68%)
